# Realistic Sky Calibration: GSM + GLEAM Closed-Loop Test

End-to-end calibration validation combining:
- **Realistic sky**: diffuse emission (Global Sky Model) + point sources (GLEAM catalog) with spectral indices
- **Measurement noise**: radiometric (thermally-limited) visibility noise
- **Beam + sky recovery**: joint optimization starting from ~30% perturbation
- **Convergence tracking**: reduced χ² metric to assess noise-weighted residuals

Key test: can we recover both sky and beam from realistic data when starting far from the truth?

In [ ]:
import os
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import healpy as hp

from astropy.time import Time
from astropy.coordinates import EarthLocation
import astropy.units as u
from astropy.table import Table

#jax.config.update("jax_enable_x64", False)
jax.config.update("jax_enable_x64", True)

from newnucal import (
    HERAArray,
    BeamModel,
    BeamBasis,
    SkyModel,
    SkyBasis,
    ForwardModel,
    Calibrator,
    apply_gains,
    init_gain_params,
)
from newnucal.basis import dpss_matrix, basis_project, basis_reconstruct
from newnucal.simulate import compute_rotation_matrices

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110})

## 1. Configuration

In [ ]:
# --- Array / observing setup ---
hexnum = 6
sep_m = 14.6

nfreq = 64
freqs = np.linspace(50e6, 225e6, nfreq)  # Hz
freq_mhz = freqs / 1e6
dnu_hz = float(np.median(np.diff(freqs)))

ntime = 8
integration_time_s = 7.5 * 60.0  # 7.5 minutes per integration

hera_loc = EarthLocation(lat=-30.7215 * u.deg, lon=21.4283 * u.deg, height=1073.0 * u.m)
t0 = Time("2023-03-21T04:00:00", scale="utc")
times = t0 + np.arange(ntime) * integration_time_s * u.s

# --- Sky / beam resolution and spectral smoothness ---
sky_nside = 32
beam_nside = 16
beam_eta_max = 20e-9
sky_eta_max = 60e-9

# --- Sky model ingredients ---
ref_freq_hz = 151e6
gleam_catalog_path = Path("../../GLEAM_EGC_v2.fits")   # edit if needed
gleam_flux_col_candidates = ["int_flux_151", "S_151", "Fint151", "peak_flux_151"]
gleam_alpha_col_candidates = ["alpha", "sp_index", "SpectralIndex"]
max_gleam_sources = 5000

# --- Noise model ---
T_rx_K = 100.0
aperture_efficiency = 0.7
noise_seed = 1234

# --- RFI injection ---
rfi_ch = 20   # channel index to contaminate

print(f"Array: {hexnum}-hex, {sep_m} m spacing")
print(f"Frequencies: {nfreq}, {freqs[0]/1e6:.0f}--{freqs[-1]/1e6:.0f} MHz")
print(f"Times: {ntime}, span {(times[-1] - times[0]).jd * 24 * 60:.1f} minutes")
print(f"Sky resolution: nside={sky_nside}, eta_max={sky_eta_max*1e9:.0f} ns")
print(f"Beam resolution: nside={beam_nside}, eta_max={beam_eta_max*1e9:.0f} ns")


## 2. Helper functions for GSM, GLEAM, and radiometric noise

In [ ]:
k_B = 1.380649e-23
c = 299792458.0

def load_gsm_temperature_maps(freqs_hz, nside):
    """Load Global Sky Model temperature maps; tries multiple pygdsm/pygsm versions."""
    gsm = None
    try:
        from pygdsm import GlobalSkyModel2016
        gsm = GlobalSkyModel2016(freq_unit="Hz")
    except Exception:
        pass
    if gsm is None:
        try:
            from pygdsm import GlobalSkyModel
            gsm = GlobalSkyModel(freq_unit="Hz")
        except Exception:
            pass
    if gsm is None:
        try:
            from pygsm import GlobalSkyModel2016
            gsm = GlobalSkyModel2016(freq_unit="Hz")
        except Exception:
            pass
    if gsm is None:
        try:
            from pygsm import GlobalSkyModel
            gsm = GlobalSkyModel(freq_unit="Hz")
        except Exception:
            pass
    if gsm is None:
        raise ImportError(
            "Could not import GSM. Install pygdsm/pygsm or edit load_gsm_temperature_maps()."
        )

    # Galactic -> Equatorial rotation
    rot_gc = hp.Rotator(coord=["G", "C"])
    maps = []
    for nu in freqs_hz:
        try:
            m_gal = gsm.generate(float(nu))
        except TypeError:
            m_gal = gsm.generate(float(nu) / 1e6)  # Some versions expect MHz
        m_gal = np.asarray(m_gal, dtype=np.float64)
        m_eq = rot_gc.rotate_map_pixel(m_gal)
        if hp.get_nside(m_eq) != nside:
            m_eq = hp.ud_grade(m_eq, nside_out=nside, power=0)
        maps.append(m_eq.astype(np.float32))
    return np.stack(maps, axis=1)  # (npix, nfreq)

def _pick_column(tab, candidates):
    for name in candidates:
        if name in tab.colnames:
            return name
    return None

def load_gleam_sources(path, max_sources=None):
    """Load GLEAM catalog, filtering for valid entries."""
    if not Path(path).exists():
        raise FileNotFoundError(
            f"GLEAM catalog not found at {path}. Point the notebook at a local GLEAM FITS file."
        )

    tab = Table.read(path)
    ra_col = _pick_column(tab, ["RAJ2000", "ra", "RA", "RA_deg"])
    dec_col = _pick_column(tab, ["DEJ2000", "dec", "DEC", "DEC_deg"])
    flux_col = _pick_column(tab, gleam_flux_col_candidates)
    alpha_col = _pick_column(tab, gleam_alpha_col_candidates)

    if ra_col is None or dec_col is None or flux_col is None:
        raise ValueError(
            f"Could not identify RA/DEC/151-MHz flux columns. Available: {tab.colnames}"
        )

    ra_deg = np.asarray(tab[ra_col], dtype=np.float64)
    dec_deg = np.asarray(tab[dec_col], dtype=np.float64)
    s151_jy = np.asarray(tab[flux_col], dtype=np.float64)

    if alpha_col is None:
        alpha = -0.8 * np.ones_like(s151_jy)
    else:
        alpha = np.asarray(tab[alpha_col], dtype=np.float64)
        alpha[~np.isfinite(alpha)] = -0.8

    # Filter to valid entries
    good = np.isfinite(ra_deg) & np.isfinite(dec_deg) & np.isfinite(s151_jy) & (s151_jy > 0)
    ra_deg, dec_deg, s151_jy, alpha = ra_deg[good], dec_deg[good], s151_jy[good], alpha[good]

    if max_sources is not None and len(s151_jy) > max_sources:
        keep = np.argsort(s151_jy)[-max_sources:]
        ra_deg, dec_deg, s151_jy, alpha = ra_deg[keep], dec_deg[keep], s151_jy[keep], alpha[keep]

    return dict(ra_deg=ra_deg, dec_deg=dec_deg, s151_jy=s151_jy, alpha=alpha)

def gleam_flux_cube_to_healpix(freqs_hz, nside, gleam, ref_freq_hz=151e6):
    """Rasterize GLEAM point sources onto HEALPix grid with frequency scaling."""
    npix = hp.nside2npix(nside)
    ra = np.asarray(gleam["ra_deg"])
    dec = np.asarray(gleam["dec_deg"])
    s151 = np.asarray(gleam["s151_jy"])
    alpha = np.asarray(gleam["alpha"])

    theta = np.deg2rad(90.0 - dec)
    phi = np.deg2rad(ra)
    pix = hp.ang2pix(nside, theta, phi)

    cube = np.zeros((npix, len(freqs_hz)), dtype=np.float32)
    scale = (freqs_hz[None, :] / ref_freq_hz) ** alpha[:, None]
    src_flux = s151[:, None] * scale
    for si, px in enumerate(pix):
        cube[px] += src_flux[si].astype(np.float32)
    return cube

def kelvin_to_jy_per_pix(temp_K, freqs_hz, nside):
    """Convert brightness temperature [K] to flux density [Jy/pix]."""
    omega_pix = 4.0 * np.pi / hp.nside2npix(nside)
    prefac = 2.0 * k_B * (freqs_hz**2) / c**2 / 1e-26  # Jy / sr / K
    return temp_K * prefac[None, :] * omega_pix

def build_gsm_plus_gleam_flux_cube(freqs_hz, nside, gleam_path, max_sources=None):
    """Combine GSM diffuse emission with GLEAM point sources."""
    gsm_temp_K = load_gsm_temperature_maps(freqs_hz, nside)
    gsm_jy = kelvin_to_jy_per_pix(gsm_temp_K, freqs_hz, nside)
    gleam = load_gleam_sources(gleam_path, max_sources=max_sources)
    gleam_jy = gleam_flux_cube_to_healpix(freqs_hz, nside, gleam)
    return gsm_temp_K, gsm_jy, gleam, gleam_jy, gsm_jy + gleam_jy

def airy_collecting_area(diameter_m, aperture_eff=0.7):
    return aperture_eff * np.pi * (diameter_m / 2.0) ** 2

def approximate_radiometric_sigma_jy(freqs_hz, gsm_temp_K, dish_diameter_m, dnu_hz, tint_s,
                                     T_rx_K=100.0, aperture_eff=0.7):
    """Estimate per-channel noise RMS from radiometric sensitivity."""
    T_sky_f = np.mean(gsm_temp_K, axis=0)
    T_sys_f = T_sky_f + T_rx_K
    A_eff = airy_collecting_area(dish_diameter_m, aperture_eff=aperture_eff)
    sefd_jy = 2.0 * k_B * T_sys_f / A_eff / 1e-26
    sigma = sefd_jy / np.sqrt(2.0 * dnu_hz * tint_s)
    return sigma.astype(np.float32), T_sys_f.astype(np.float32)

def complex_gaussian_noise(shape, sigma_f, rng):
    """Generate complex Gaussian noise with frequency-dependent amplitude."""
    sigma = np.asarray(sigma_f, dtype=np.float32)[None, :, None]
    nre = rng.standard_normal(shape).astype(np.float32)
    nim = rng.standard_normal(shape).astype(np.float32)
    return sigma * (nre + 1j * nim) / np.sqrt(2.0)

## 3. Array, beam, times, and rotation matrices

In [ ]:
array = HERAArray.from_hex(hexnum=hexnum, sep=sep_m)
print(f"Antennas: {array.nants},  Baselines: {array.nbls}")

rot_matrices = compute_rotation_matrices(times, hera_loc)
print(f"rot_matrices shape: {rot_matrices.shape}")

# Load Airy SVD beam basis (18 modes) — matches closed_loop_test.
beam_basis = BeamBasis.from_file('beam_basis_airy.npz', new_freqs=freqs)
beam_model = BeamModel(nside=beam_nside, freqs=freqs, basis=beam_basis)
print(f"Beam SVD modes: {beam_model.A.shape[1]}")

## 4. Build the GSM + GLEAM sky and project to DPSS basis

In [ ]:
# Cache the GSM + GLEAM cube to avoid the slow pygdsm/healpy rotation on every run.
# Cache is keyed by (freqs, sky_nside, gleam_path, max_sources); invalidated if any change.
sky_cache_path = Path(f"sky_cube_cache_nside{sky_nside}_nfreq{nfreq}_n{max_gleam_sources}.npz")
cache_ok = False
if sky_cache_path.exists():
    c = np.load(sky_cache_path, allow_pickle=False)
    cache_ok = (
        c["freqs_hz"].shape == freqs.shape
        and np.allclose(c["freqs_hz"], freqs)
        and int(c["sky_nside"]) == sky_nside
        and int(c["max_gleam_sources"]) == max_gleam_sources
        and str(c["gleam_path"]) == str(gleam_catalog_path)
    )

if cache_ok:
    print(f"Loading sky models from cache: {sky_cache_path}")
    gsm_temp_K = c["gsm_temp_K"]
    gsm_jy     = c["gsm_jy"]
    gleam_jy   = c["gleam_jy"]
    flux_true  = c["flux_true"]
    gleam = {
        "ra_deg":   c["gleam_ra_deg"],
        "dec_deg":  c["gleam_dec_deg"],
        "s151_jy":  c["gleam_s151_jy"],
        "alpha":    c["gleam_alpha"],
    }
else:
    print("Building GSM + GLEAM cube (slow; will cache to disk for next run)...")
    gsm_temp_K, gsm_jy, gleam, gleam_jy, flux_true = build_gsm_plus_gleam_flux_cube(
        freqs,
        sky_nside,
        gleam_catalog_path,
        max_sources=max_gleam_sources,
    )
    np.savez(
        sky_cache_path,
        freqs_hz=np.asarray(freqs, dtype=np.float64),
        sky_nside=np.int32(sky_nside),
        max_gleam_sources=np.int32(max_gleam_sources),
        gleam_path=str(gleam_catalog_path),
        gsm_temp_K=gsm_temp_K,
        gsm_jy=gsm_jy,
        gleam_jy=gleam_jy,
        flux_true=flux_true,
        gleam_ra_deg=np.asarray(gleam["ra_deg"]),
        gleam_dec_deg=np.asarray(gleam["dec_deg"]),
        gleam_s151_jy=np.asarray(gleam["s151_jy"]),
        gleam_alpha=np.asarray(gleam["alpha"]),
    )
    print(f"Cached to {sky_cache_path}")

print(f"GSM temperature cube: {gsm_temp_K.shape}")
print(f"GLEAM sources used:   {len(gleam['s151_jy'])}")
print(f"Combined flux cube:   {flux_true.shape}  [Jy/pixel]")
print(f"  min/max flux: {flux_true.min():.2e} -- {flux_true.max():.2e} Jy")
print(f"  GLEAM spectral indices: min/mean/max = {gleam['alpha'].min():.2f} / {gleam['alpha'].mean():.2f} / {gleam['alpha'].max():.2f}")

# Load GSM+GLEAM SVD sky basis (6 modes) — matches closed_loop_test.
# Far better-conditioned than DPSS at this dynamic range; the SVD modes
# are tuned to the actual GSM+GLEAM spectral structure.
sky_basis = SkyBasis.from_file('sky_basis_gsm_gleam.npz', new_freqs=freqs)
A_sky = sky_basis.A  # Keep reference for reconstructions
print(f"Sky SVD modes: {sky_basis.nmodes}")

# Project to sky coefficients
sky_coeffs_true = jnp.array(basis_project(flux_true, A_sky), dtype=jnp.float32)
print(f"sky_coeffs_true shape: {sky_coeffs_true.shape}")

## 5. Create sky model and simulate true visibilities with gains + noise

In [ ]:
# Create SkyModel for ForwardModel
sky_model = SkyModel(nside=sky_nside, freqs=freqs, basis=sky_basis)

# Create ForwardModel
fwd = ForwardModel(array, sky_model, beam_model, freqs, eps=1e-5)

print("Simulating noiseless visibilities (compiles JIT on first run)...")
vis_true = fwd.simulate(sky_coeffs_true, jnp.array(rot_matrices))
print(f"vis_true shape: {vis_true.shape}")
print(f"Mean |vis_true|: {float(jnp.abs(vis_true).mean()):.4e}")

# Smooth gain perturbations (time and frequency variation)
t_norm = np.linspace(0, 1, ntime)[:, None]
f_norm = np.linspace(0, 1, nfreq)[None, :]

true_log_amp = (
    0.05 * np.cos(2 * np.pi * f_norm)
    + 0.02 * np.sin(2 * np.pi * t_norm)
).astype(np.float32)

true_phase = (
    0.15 * np.sin(2 * np.pi * f_norm)
    + 0.05 * np.cos(2 * np.pi * t_norm)
).astype(np.float32)

true_phi = np.zeros((ntime, 2, nfreq), dtype=np.float32)
true_phi[:, 0, :] = 1e-4 * np.cos(2 * np.pi * f_norm) + 3e-5 * np.sin(2 * np.pi * t_norm)
true_phi[:, 1, :] = 5e-5 * np.sin(2 * np.pi * f_norm) + 2e-5 * np.cos(2 * np.pi * t_norm)

# Apply gains
vis_gain_only = apply_gains(
    vis_true,
    jnp.array(true_log_amp),
    jnp.array(true_phase),
    jnp.array(true_phi),
    jnp.array(array.bls, dtype=jnp.float32),
)

# Radiometric noise
sigma_vis_f, Tsys_f = approximate_radiometric_sigma_jy(
    freqs,
    gsm_temp_K,
    dish_diameter_m=14.6,
    dnu_hz=dnu_hz,
    tint_s=integration_time_s,
    T_rx_K=T_rx_K,
    aperture_eff=aperture_efficiency,
)

rng = np.random.default_rng(noise_seed)
noise = complex_gaussian_noise(vis_gain_only.shape, sigma_vis_f, rng)
vis_clean = jnp.array(np.array(vis_gain_only + jnp.array(noise)), dtype=jnp.complex64)  # pre-RFI data
vis_data  = np.array(vis_clean)
noise_sigma = jnp.array(sigma_vis_f, dtype=jnp.float32)[None, :, None]

# Inject narrowband RFI — amplitude = mean |vis| at that channel so spike is visible
rfi_amplitude = float(jnp.abs(vis_gain_only[:, rfi_ch, :]).mean())
vis_data_np = np.array(vis_data)
vis_data_np[:, rfi_ch, :] += rfi_amplitude * (1.0 + 0.3j)
vis_data = jnp.array(vis_data_np, dtype=jnp.complex64)

print(f"Noise RMS: {sigma_vis_f.min():.3e} -- {sigma_vis_f.max():.3e} Jy")
print(f"System temp: {Tsys_f.min():.0f} -- {Tsys_f.max():.0f} K")
print(f"RFI ch {rfi_ch} ({freqs[rfi_ch]/1e6:.1f} MHz): amplitude {rfi_amplitude:.3e} Jy  ({rfi_amplitude/sigma_vis_f[rfi_ch]:.0f}x noise RMS)")
print(f"Data shape: {vis_data.shape}")

## 6. Initial diagnostic plots

In [ ]:
ref_ifreq = int(np.argmin(np.abs(freqs - ref_freq_hz)))

fig = plt.figure(figsize=(12, 9))
hp.mollview(gsm_jy[:, ref_ifreq], fig=fig, sub=(3, 1, 1),
            title=f"Diffuse GSM ({freqs[ref_ifreq]/1e6:.0f} MHz)", cmap="inferno")
hp.mollview(gleam_jy[:, ref_ifreq], fig=fig, sub=(3, 1, 2),
            title="Point sources (GLEAM rasterized)", cmap="inferno")
hp.mollview(flux_true[:, ref_ifreq], fig=fig, sub=(3, 1, 3),
            title="Combined (GSM + GLEAM)", cmap="inferno")
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(freq_mhz, sigma_vis_f, "o-", label="Noise RMS")
ax.set_xlabel("Frequency [MHz]")
ax.set_ylabel("Visibility noise RMS [Jy]")
ax.set_title("Radiometric noise per channel")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(freq_mhz, Tsys_f, "s-", label="System temperature")
ax.set_xlabel("Frequency [MHz]")
ax.set_ylabel("Temperature [K]")
ax.set_title("System temperature (sky + receiver)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Setup calibrator and define reduced χ²

In [ ]:
cal = Calibrator(
    array=array,
    beam_model=beam_model,
    sky_model=sky_model,
    freqs=freqs,
    rot_matrices=rot_matrices,
    data=vis_data,
    eps=1e-5,
)

# Set the calibrator's internal noise model to the actual radiometric sigma.
# With float64 there's no precision argument for the earlier vis_scale hack,
# and using the real sigma makes the internal loss the proper noise-weighted
# chi^2 (same metric as reduced_chi2 below). Per-channel weighting also lets
# the optimizer prioritize information-rich channels.
cal.set_noise_sigma(jnp.array(sigma_vis_f, dtype=jnp.float64))

beam_coeffs_true = jnp.array(beam_model.coeffs, dtype=jnp.float64)

def reduced_chi2(cal, params, sigma_vis, data=None, exclude_chans=None):
    """Noise-weighted reduced chi2.
    data          : override cal.data (e.g. vis_clean for truth check).
    exclude_chans : list of channel indices to omit (e.g. [rfi_ch]).
    """
    if data is None:
        data = cal.data
    resid = cal.simulate(params) - data
    sv = sigma_vis
    if exclude_chans is not None:
        keep = np.ones(resid.shape[1], dtype=bool)
        keep[list(exclude_chans)] = False
        resid = resid[:, keep, :]
        sv    = sv   [:, keep, :]
    chi2 = jnp.sum(jnp.abs(resid) ** 2 / sv ** 2)
    return float(chi2 / resid.size)

# Sanity check on non-RFI channels vs clean data — should be ~1.0
params_truth = {
    "sky_coeffs": sky_coeffs_true,
    "beam_coeffs": beam_coeffs_true,
    "log_amp": jnp.array(true_log_amp),
    "phase":   jnp.array(true_phase),
    "phi":     jnp.array(true_phi),
}
chi2_truth = reduced_chi2(cal, params_truth, noise_sigma,
                          data=vis_clean, exclude_chans=[rfi_ch])
print(f"Truth reduced chi2 (non-RFI channels, vs clean data, should be ~1.0): {chi2_truth:.4f}")

## 8. Apply sky and beam masks

In [ ]:
beam_mask = cal.build_beam_mask_altitude(0)
sky_mask  = cal.build_sky_mask_from_beam_pixels(beam_mask)
cal.apply_sky_mask(sky_mask)
cal.apply_beam_mask(beam_mask)
print(f"Sky pixels retained:  {int(sky_mask.sum())} / {len(sky_mask)}")
print(f"Beam pixels retained: {int(beam_mask.sum())} / {len(beam_mask)}")

## 9. Perturb sky and beam to create starting point (30% perturbation)

In [ ]:
rng_perturb = np.random.default_rng(99)

sky_coeffs_perturbed = sky_coeffs_true + 0.30 * jnp.array(
    rng_perturb.standard_normal(sky_coeffs_true.shape).astype(np.float32)
) * float(jnp.abs(sky_coeffs_true).mean())

beam_coeffs_perturbed = beam_coeffs_true + 0.30 * jnp.array(
    rng_perturb.standard_normal(beam_coeffs_true.shape).astype(np.float32)
) * float(jnp.abs(beam_coeffs_true).mean())

prms0 = {
    "sky_coeffs":     sky_coeffs_perturbed,
    "beam_coeffs":    beam_coeffs_perturbed,
    "log_ch_weights": jnp.zeros(nfreq, dtype=jnp.float32),
    **init_gain_params(ntime, nfreq),
}

loss_init = cal.calc_loss(prms0)
# Exclude the RFI channel so chi2_init is comparable to chi2_final
chi2_init = reduced_chi2(cal, prms0, noise_sigma, exclude_chans=[rfi_ch])
print(f"Initial loss (calibrator units): {loss_init:.4e}")
print(f"Initial reduced chi2 (non-RFI channels): {chi2_init:.4e}  (truth: {chi2_truth:.4f})")
print(f"Improvement needed: {chi2_init / chi2_truth:.1f}x")

In [ ]:
prms = prms0.copy()

## 10. Run joint sky + beam + gain recovery

In [ ]:
print(f"Starting reduced chi2 (non-RFI channels): {chi2_init:.4e}")
print("="*70)
print("Joint sky + beam + gain recovery")
print("="*70)

prms, loss = cal.fit_joint_sky_beam_dirty(
    prms,
    n_iter=40,
    sky_beam_reg=3e-3,
    #joint_initial_step=1.2,
    joint_initial_step=[0.1, 1.0, 10.0],
    solve_every=None,
    joint_anderson_history=2,
    rfi_smooth_width_chans=5,
    rfi_log_threshold=np.log(3.0),
    rfi_gamma=3.00,
    rfi_log_min_weight=np.log(1e-10),
    rfi_alpha_down=0.9,
    rfi_alpha_up=0.9,
    rfi_min_retention_per_update=0.1,
    max_rfi_updates=None,
    verbose=True,
)

chi2_final = reduced_chi2(cal, prms, noise_sigma, exclude_chans=[rfi_ch])
print(f"\nFinal loss (calibrator units): {loss:.4e}")
print(f"Final reduced chi2 (non-RFI channels): {chi2_final:.4e}")
print(f"Improvement: {chi2_init / chi2_final:.2f}x")

## 10b. Block-coordinate L-BFGS refinement (Calibrator method)

`cal.fit_joint_sky_beam_lbfgs(...)` is the L-BFGS counterpart to `fit_joint_sky_beam_dirty`. Each outer iteration alternates three well-conditioned subproblems:

1. **Gains** via closed-form linear solve (`fit_gains_linear_variable_beam`).
2. **RFI channel weights** via `fit_channel_weights_local_chi2_exponential`.
3. **Sky + beam** via `jaxopt.LBFGS` with per-block gradient-norm preconditioning. Gains and RFI weights are passed as pytree args (not closure-captured) so repeated calls do *not* trigger JIT recompilation.

`project_degeneracies=True` enforces the canonical gauge `mean_t(log_amp) = 0` after each outer iteration, absorbing the time-mean amplitude into `beam_coeffs`. Same idea as `hera_cal.redcal` degeneracy projection — preserves the forward visibilities but moves parameters into a comparable-to-truth gauge.

The first call JIT-compiles; subsequent calls (with the same `lbfgs_*` config) reuse the cached solver. `cal._recompile_jit()` (called when masks change) invalidates the cache.


In [ ]:
prms, loss = cal.fit_joint_sky_beam_lbfgs(
    prms,
    n_outer=3,
    lbfgs_maxiter=200,
    lbfgs_history_size=30,
    rfi_passes_per_outer=5,
    rfi_smooth_width_chans=5,
    rfi_log_threshold=np.log(3.0),
    rfi_gamma=3.00,
    rfi_log_min_weight=np.log(1e-10),
    rfi_alpha_down=0.9,
    rfi_alpha_up=0.9,
    rfi_min_retention_per_update=0.1,
    project_degeneracies=True,
    verbose=True,
)

chi2_after = reduced_chi2(cal, prms, noise_sigma, exclude_chans=[rfi_ch])
print(f"\nFinal: loss={loss:.4e}  ext chi2={chi2_after:.4e}")

## 11. Convergence summary

In [ ]:
print("Convergence summary (non-RFI channels):")
print(f"  Truth  chi2_r: {chi2_truth:.4f}")
print(f"  Initial chi2_r: {chi2_init:.4e}")
print(f"  Final   chi2_r: {chi2_final:.4e}")
print(f"  Improvement:    {chi2_init / chi2_final:.2f}x")
print()
print("Re-run fit cell to continue from current prms (hot-start).")
print("Re-run 'prms = prms0.copy()' to restart from the perturbed initial point.")

## 12. Residual analysis: baseline spectra

In [ ]:
vis_init = cal.simulate(prms0)    # model at perturbed start
vis_fit  = cal.simulate(prms)     # model after fitting
resid = vis_data - vis_fit        # final residuals (used by later cells)

bl_indices = [0, array.nbls // 4, array.nbls // 2, array.nbls - 1]
t_show = 0
bls_j = jnp.array(array.bls, dtype=jnp.float32)

fig, axes = plt.subplots(len(bl_indices), 2, figsize=(13, 2.8 * len(bl_indices)), sharex=True)
fig.suptitle("Recovered visibilities vs. data (RFI channel marked)", fontsize=12)

for row, bi in enumerate(bl_indices):
    bl_len = float(jnp.linalg.norm(bls_j[bi, :2]))

    ax = axes[row, 0]
    ax.semilogy(freq_mhz, jnp.abs(vis_data)[t_show, :, bi], "k-",  lw=1.5, label="Data", alpha=0.7)
    ax.semilogy(freq_mhz, jnp.abs(vis_init)[t_show, :, bi], "b--", lw=1.0, label="Initial model", alpha=0.6)
    ax.semilogy(freq_mhz, jnp.abs(vis_fit) [t_show, :, bi], "r--", lw=1.5, label="Recovered model")
    ax.semilogy(freq_mhz, jnp.abs(resid)   [t_show, :, bi], "m:",  lw=1.2, label="Residual")
    ax.semilogy(freq_mhz, noise_sigma[0, :, 0], color="gray", ls=":", lw=1, label="Noise RMS")
    ax.axvline(freqs[rfi_ch]/1e6, color="orange", ls=":", lw=2, alpha=0.8)
    ax.set_ylabel(f"bl {bi}\n({bl_len:.0f}m) |V|")
    if row == 0:
        ax.legend(fontsize=7, loc="upper right", ncol=2)
    if row == len(bl_indices) - 1:
        ax.set_xlabel("Frequency [MHz]")

    ax = axes[row, 1]
    ax.plot(freq_mhz, jnp.angle(vis_data)[t_show, :, bi], "k-",  lw=1.5, label="Data", alpha=0.7)
    ax.plot(freq_mhz, jnp.angle(vis_fit) [t_show, :, bi], "r--", lw=1.5, label="Recovered model")
    ax.axvline(freqs[rfi_ch]/1e6, color="orange", ls=":", lw=2, alpha=0.8)
    ax.set_ylabel(f"bl {bi}\narg(V) [rad]")
    if row == 0:
        ax.legend(fontsize=7, loc="upper right")
    if row == len(bl_indices) - 1:
        ax.set_xlabel("Frequency [MHz]")

plt.tight_layout()
plt.show()

## 13. Per-channel residual diagnostics

In [ ]:
# Channel-averaged residual power
resid_power = jnp.abs(resid) ** 2
resid_power_f = jnp.mean(resid_power, axis=(0, 2))  # Average over time, baseline
data_power_f = jnp.mean(jnp.abs(vis_data) ** 2, axis=(0, 2))

# Per-channel chi2
chi2_by_channel = jnp.sum(resid_power, axis=(0, 2)) / (noise_sigma[0, :, 0] ** 2)
chi2_dof_by_channel = chi2_by_channel / (ntime * array.nbls)

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# Residual power
ax = axes[0]
ax.semilogy(freq_mhz, data_power_f, "k-", lw=1.5, label="Data power", alpha=0.7)
ax.semilogy(freq_mhz, resid_power_f, "r-", lw=1.5, label="Residual power")
ax.set_ylabel("Power [Jy²]")
ax.set_title("Per-channel residual power: channels with large residuals need more work")
ax.grid(True, alpha=0.3)
ax.legend()

# Chi2 per channel
ax = axes[1]
ax.semilogy(freq_mhz, chi2_dof_by_channel, "b-", lw=2, marker="o", markersize=4)
ax.axhline(1.0, color="k", ls="--", lw=1.5, alpha=0.5, label="Ideal (1.0)")
ax.set_xlabel("Frequency [MHz]")
ax.set_ylabel("Reduced χ² (per channel)")
ax.set_title("Reduced χ² by frequency: should be close to 1 everywhere")
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_ylim([0.5, 2000.0])

plt.tight_layout()
plt.show()

print(f"Per-channel χ²_ν statistics:")
print(f"  min:  {float(chi2_dof_by_channel.min()):.4f}")
print(f"  mean: {float(chi2_dof_by_channel.mean()):.4f}")
print(f"  max:  {float(chi2_dof_by_channel.max()):.4f}")

## 14. Sky recovery diagnostics

In [ ]:
# Reconstruct sky maps
sky_rec_full = np.array(basis_reconstruct(prms["sky_coeffs"], A_sky))  # (npix, nfreq)
sky_rec = cal.pixel_mask[:, None] * sky_rec_full   # zero out unoptimized pixels

# RMS on solved pixels only
mask_bool = cal.pixel_mask.astype(bool)
sky_diff  = sky_rec_full[mask_bool] - flux_true[mask_bool]  # (n_solved, nfreq)
rms_error = float(np.sqrt(np.mean(sky_diff ** 2)))
rms_true  = float(np.sqrt(np.mean(flux_true[mask_bool] ** 2)))
print(f"Sky recovery (solved pixels only):")
print(f"  RMS error:      {rms_error:.4e} Jy")
print(f"  RMS true:       {rms_true:.4e} Jy")
print(f"  Relative error: {rms_error/rms_true:.4f}")

fig = plt.figure(figsize=(13, 10))
vmax_sky = np.percentile(flux_true[mask_bool, ref_ifreq], 99)
diff_scale = np.percentile(np.abs(sky_diff[:, ref_ifreq]), 95)
hp.mollview(flux_true[:, ref_ifreq], fig=fig, sub=(3,1,1),
            title=f"True sky (GSM+GLEAM, {freqs[ref_ifreq]/1e6:.0f} MHz)",
            min=0, max=vmax_sky, cmap="inferno")
hp.mollview(sky_rec[:, ref_ifreq], fig=fig, sub=(3,1,2),
            title="Recovered sky (solved pixels)",
            min=0, max=vmax_sky, cmap="inferno")
hp.mollview((sky_rec_full - flux_true)[:, ref_ifreq] * cal.pixel_mask, fig=fig, sub=(3,1,3),
            title="Residual (recovered - true, solved pixels)",
            min=-diff_scale, max=diff_scale, cmap="RdBu_r")
plt.show()

## 15. Beam recovery diagnostics (gauge-aligned to truth)

In [ ]:
# Gauge-align recovered into truth's gain gauge for clearer comparison.
# The fit's project_degeneracies leaves the solution in canonical
# mean_t(log_amp) = 0 gauge, while truth has
#   mean_t(true_log_amp[t,f]) = 0.05·cos(2π·f_norm[f])
# by construction. Shift recovered into truth's gauge via the
# visibility-preserving transformation:
#   log_amp_new = log_amp + mean_t(true_log_amp)
#   beam_new    = beam    / exp(mean_t(true_log_amp))
mean_log_amp_true = np.mean(true_log_amp, axis=0)              # (nfreq,)
gauge_scale_f     = np.exp(mean_log_amp_true)[None, :]         # (1, nfreq)

beam_rec_canon = np.array(basis_reconstruct(prms["beam_coeffs"], beam_model.A))
beam_rec       = beam_rec_canon / gauge_scale_f                # gauge-shifted to truth
beam_true_spec = np.array(basis_reconstruct(beam_coeffs_true,   beam_model.A))

beam_diff = beam_rec - beam_true_spec
rms_beam  = float(np.sqrt(np.mean(beam_diff ** 2)))
rms_sig   = float(np.sqrt(np.mean(beam_true_spec ** 2)))

# Apparent (gain × beam) is gauge-invariant — same number either gauge.
rec_log_amp_truth_gauge = np.array(prms["log_amp"]) + mean_log_amp_true[None, :]
true_apparent = np.exp(true_log_amp[:, None, :])           * beam_true_spec
rec_apparent  = np.exp(rec_log_amp_truth_gauge[:, None, :]) * beam_rec
print(f"Beam recovery (gauge-aligned to truth):")
print(f"  RMS error:      {rms_beam:.4e}")
print(f"  RMS true:       {rms_sig:.4e}")
print(f"  Relative error: {rms_beam/rms_sig:.4f}")
print(f"  Apparent relative error: {np.sqrt(np.mean((rec_apparent - true_apparent) ** 2)) / np.sqrt(np.mean(true_apparent ** 2)):.4f}")


fig = plt.figure(figsize=(13, 10))
vmax_b = np.percentile(beam_true_spec[:, ref_ifreq], 99)
hp.orthview(beam_true_spec[:, ref_ifreq], rot=(0,90,0), half_sky=True, fig=fig, sub=(3,1,1),
            title=f"True beam ({freqs[ref_ifreq]/1e6:.0f} MHz)", min=0, max=vmax_b, cmap="inferno")
hp.orthview(beam_rec[:, ref_ifreq],       rot=(0,90,0), half_sky=True, fig=fig, sub=(3,1,2),
            title="Recovered beam (gauge-aligned to truth)", min=0, max=vmax_b, cmap="inferno")
dscale = np.percentile(np.abs(beam_diff[:, ref_ifreq]), 95)
hp.orthview(beam_diff[:, ref_ifreq],      rot=(0,90,0), half_sky=True, fig=fig, sub=(3,1,3),
            title="Residual (recovered - true)", min=-dscale, max=dscale, cmap="RdBu_r")
plt.show()

peak_px = int(np.argmax(beam_true_spec[:, ref_ifreq]))
npix_b  = beam_true_spec.shape[0]
sample_pxs = [peak_px,
               int(np.argsort(beam_true_spec[:, ref_ifreq])[-npix_b//4]),
               int(np.argsort(beam_true_spec[:, ref_ifreq])[-npix_b//2])]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("Beam spectra: true vs. recovered (gauge-aligned)", fontsize=11)
for ax, px in zip(axes, sample_pxs):
    ax.plot(freq_mhz, beam_true_spec[px], "k-",  lw=2,   label="True")
    ax.plot(freq_mhz, beam_rec[px],       "r--", lw=1.5, label="Recovered")
    ax.set_title(f"pixel {px}")
    ax.set_xlabel("Frequency [MHz]"); ax.set_ylabel("Beam amplitude")
    ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


## 16. Gain recovery (gauge-aligned to truth)

In [ ]:
# Gauge-align recovered log_amp into truth's gauge (see cell 34 for the
# transformation). Recovered phase doesn't have the same gauge issue since
# the fit doesn't currently project a phase degeneracy.
mean_log_amp_true = np.mean(true_log_amp, axis=0)
rec_log_amp = np.array(prms["log_amp"]) + mean_log_amp_true[None, :]
rec_phase   = np.array(prms["phase"])

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, data, title in zip(axes,
        [true_log_amp, rec_log_amp, rec_log_amp - true_log_amp],
        ["True log amplitude", "Recovered log amplitude (gauge-aligned)", "Residual (recovered - true)"]):
    im = ax.pcolormesh(freq_mhz, np.arange(ntime), data, cmap="RdBu_r", shading="auto")
    ax.set_xlabel("Frequency [MHz]"); ax.set_ylabel("Time index"); ax.set_title(title)
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()
print(f"Log amplitude RMS error (gauge-aligned): {float(np.sqrt(np.mean((rec_log_amp - true_log_amp)**2))):.4e}")


## 17. Channel weight diagnostics (RFI)

In [ ]:
ch_weights = np.exp(np.array(prms["log_ch_weights"]))
# Calibrator may expand to (ntime, nfreq); average over time for display
ch_weights_f = ch_weights.mean(axis=0) if ch_weights.ndim == 2 else ch_weights

fig, ax = plt.subplots(figsize=(12, 4))
ax.semilogy(freq_mhz, ch_weights_f, "go-", lw=2, ms=4, label="Channel weights (time-averaged)")
ax.axvline(freqs[rfi_ch]/1e6, color="orange", ls=":", lw=2.5, label=f"RFI channel {rfi_ch}")
ax.axhline(1.0, color="gray", ls="--", lw=1, alpha=0.5)
ax.set_xlabel("Frequency [MHz]"); ax.set_ylabel("Channel weight")
ax.set_title("RFI-optimized channel weights: contaminated channel is downweighted")
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim([1e-6, 2])
plt.tight_layout(); plt.show()

print(f"RFI channel {rfi_ch} ({freqs[rfi_ch]/1e6:.1f} MHz) weight: {ch_weights_f[rfi_ch]:.4e}")
print(f"Neighbour weights: ch{rfi_ch-1}={ch_weights_f[rfi_ch-1]:.4f}  ch{rfi_ch+1}={ch_weights_f[rfi_ch+1]:.4f}")
print(f"Median non-RFI weight: {float(np.median(np.delete(ch_weights_f, rfi_ch))):.4f}")

## Summary

End-to-end calibration recovery from realistic data:

1. **Realistic sky**: Global Sky Model (diffuse) + GLEAM catalog (point sources) with per-source spectral indices
2. **Measurement noise**: Radiometric sensitivity based on system temperature and integration time
3. **RFI injection**: Narrowband spike (1000× noise RMS) in channel 20, recovered via soft downweighting
4. **Sky/beam masking**: Below-horizon pixels removed before optimization
5. **Significant perturbation**: 30% Gaussian perturbation on both sky and beam coefficients

**Key results:**
- Reduced χ² converges toward 1.0 over the 20-iteration joint fit
- RFI channel is identified and heavily downweighted by the optimizer
- Sky and beam maps are recovered on above-horizon pixels with manageable residuals
- Gains show expected frequency and time structure

**Physics being tested:**
- DPSS spectral constraints (sky and beam) limit degeneracies in frequency
- Earth rotation breaks sky/beam degeneracies over time
- Joint optimization simultaneously recovers sky, beam, gains, and RFI weights